# **Stage 2 (benchmark) — Delinquency Rate: Multivariate Time-Series Forecast**
## SARIMAX · ARDL · VAR | US Quarterly
___
**Purpose.** Forecast the US delinquency rate (`us_delinquency_rate`) directly with
**time-series methods**, as an interpretable, audit-defensible **benchmark** for the
machine-learning PD model built by the wider team. This notebook is the
time-series counterpart to that ML work — the ML methods are deliberately **out of
scope here**.

**How this consumes Stage 1 (02a).** The single input is `SARIMA_regressors_US_Q.csv`
— the *output* of `02a_Stage1_TimeSeries.ipynb`. That one file already contains both
(i) the historical delinquency target and (ii) the **forecast-period paths** of every
macro regressor. So 02a's forecasts are used exactly as intended: as the **future
exogenous inputs** that the conditional models (SARIMAX, ARDL) need to project the
target beyond the last observation.

| Step | Description | Output |
|------|-------------|--------|
| 0 | Configuration & data loading (`Stage1_final_regressors_US_Q.csv`) | — |
| 1 | Sample diagnostics & EDA (COVID transmission break) | — |
| 2 | Model specifications (orders selected once) | — |
| 3 | Out-of-sample backtest + Diebold-Mariano | LaTeX table *(disk export off by default)* |
| 3b | Specification-search robustness + placebo validation | LaTeX table *(disk export off by default)* |
| 4 | 20-quarter forward forecast (2026–2030) | — |
| 5 | Export & limitations | `TS_delinquency_forecast_US_Q.csv` *(disk export off by default)* |

**Models.** Naive random walk (DM benchmark) · **AR(1)** (requested baseline) ·
SARIMA (best univariate) · **SARIMAX** and **ARDL** (macro-conditional — these consume
02a's exogenous forecasts) · **VAR** (joint-system cross-check).

---
### Methodological notes — read before interpreting results

1. **"Multivariate" splits into two families that treat 02a differently.**
   *Single-equation conditional* models (SARIMAX, ARDL) take the other variables as
   given inputs, so they genuinely consume 02a's forecasts. *Joint-system* models
   (VAR, and its VECM/BVAR/FAVAR relatives) forecast **all** variables simultaneously
   from their own dynamics — they generate their own macro paths and do **not** consume
   02a. The VAR here is therefore an independent cross-check, not a consumer of Stage 1;
   forcing 02a's paths into it would require hard-conditional forecasting and change the
   model's character. SARIMAX/ARDL are the primary models that match the intended design.

2. **The conditional intervals understate uncertainty.** SARIMAX/ARDL treat 02a's
   forecasted regressors as *known*. The real forecast also carries the uncertainty of
   those upstream macro forecasts, which is not propagated here. Interval bands are shown
   for the conditional mean only; full uncertainty quantification and coherent IFRS 9
   base/upside/downside construction are a **later stage** and are not attempted here.

3. **COVID is excluded from both fitting and evaluation.** During 2020–21, forbearance
   severed the macro→credit link: delinquency *fell* to ~1.5% while unemployment spiked
   to 11%. A macro-conditional model cannot (and should not) explain this. COVID quarters
   are set to `NaN` (state-space handles missing data natively — the gold-standard
   treatment noted in 02a) and every backtest origin whose target lands in the window is
   skipped.

4. **A one-step backtest does not fully validate the 5-year path.** At `h=1` every lagged
   regressor is already realised, giving a clean, fair test. The 20-quarter forward
   forecast, by contrast, leans on 02a's *forecasted* regressors once the horizon exceeds
   each variable's lag. Read the backtest as validating short-horizon conditional skill,
   not the full lifetime path.

*Scope note:* VECM, Bayesian VAR and FAVAR were considered and left out for parsimony at
this sample size (~130 usable quarters) and for audit interpretability; they can be added
if a joint-system extension is wanted.


**References:**
Box, G.E.P. & Jenkins, G.M. (1976). *Time Series Analysis: Forecasting and Control.* Holden-Day.
Diebold, F.X. & Mariano, R.S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3), 253–263.
Harvey, D., Leybourne, S. & Newbold, P. (1997). Testing the equality of prediction mean squared errors. *International Journal of Forecasting*, 13(2), 281–291.
Clark, T.E. & West, K.D. (2007). Approximately normal tests for equal predictive accuracy in nested models. *Journal of Econometrics*, 138(1), 291–311.


## **0: Configuration & Data Loading**
___
All user-facing settings live here — target, exogenous block, COVID window,
horizons and output filename. Data is loaded directly from the project GitHub repo
(the same repo Stage 1 writes to), with a local fallback to `../Data Collection/`.

Two export toggles — `SAVE_LATEX_TABLE` and `SAVE_CSV_OUTPUTS` — are set to `False`
by default: every graph, table and chart in this notebook renders inline only. Flip
either flag to `True` once you're ready to also write results back to disk.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import logging; logging.getLogger('statsmodels').setLevel(logging.ERROR)
from statsmodels.tools.sm_exceptions import ConvergenceWarning, ValueWarning
warnings.simplefilter('ignore', ConvergenceWarning)   # expected during rolling re-fits
warnings.simplefilter('ignore', ValueWarning)         # date-index freq notices
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pmdarima import auto_arima
from IPython.display import display

# ── GitHub data source (same repo as Stage 1 / 02a) ───────────────────────────
GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/hogandan85/ST-498/main/Data%20Collection'
GITHUB_TOKEN    = None
LOCAL_OUTPUT    = Path.cwd()/ 'Data Collection'

# ── Input: the Stage 1 (02a) output ───────────────────────────────────────────
# Contains the historical target AND the forecast-period paths of every macro
# regressor, so 02a's forecasts serve as the future exogenous inputs below.
INPUT_REGRESSORS = 'Stage1_final_regressors_US_Q.csv'   # 02c winner-selection output
                                                        # (was SARIMA_regressors_US_Q.csv)

# ── Target & exogenous blocks ─────────────────────────────────────────────────
TARGET = 'us_delinquency_rate'
# Six CCF-significant LAGGED predictors (identical to the Stage 2 regressor set).
# Because each is lagged, at h=1 every value is already realised -> the one-step
# backtest uses no forecasted inputs and is a fair test.
# NOTE: two lags were re-derived upstream in 02c — us_unemployment L1->L0 and
# us_gdp_yoy_growth L0->L2. The names below match Stage1_final_regressors_US_Q.csv.
EXOG = ['us_house_price_yoy_L3', 'us_consumer_confidence_L2', 'us_unemployment_L0',
        'us_credit_qoq_growth_L6', 'us_gdp_yoy_growth_L2', 'us_cpi_L6']
# Contemporaneous block for the VAR cross-check. A VAR forecasts all of these
# jointly from their own dynamics -> it does NOT consume 02a (see notes in Cell 0).
VAR_VARS = [TARGET, 'us_unemployment', 'us_gdp_yoy_growth', 'us_cpi', 'us_credit_qoq_growth']

# ── COVID exclusion window (closed interval; both endpoints excluded) ──────────
COVID_START, COVID_END = '2020-03-31', '2021-12-31'

# ── Horizons ──────────────────────────────────────────────────────────────────
M          = 4      # seasonal period (quarterly)
H          = 20     # forward horizon (5 years; IFRS 9 lifetime)
BACKTEST_H = 1      # OOS evaluation horizon (1 => cleanest DM; lagged exog all realised)
BACKTEST_MIN_TRAIN = 60

# ── Colour palette (matches 02a) ──────────────────────────────────────────────
NAVY, BLUE, RED, AMBER, GREEN, GREY = (
    '#1F3864', '#2E75B6', '#C0392B', '#E67E22', '#27AE60', '#7F8C8D')
TEMPLATE = 'plotly_white'
RECESSIONS_US = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2001-03-01', '2001-12-31', 'Dot-com'),
    ('2007-12-01', '2009-06-30', 'GFC'),
    ('2020-02-01', '2020-04-30', 'COVID'),
]

# ── Output filename ───────────────────────────────────────────────────────────
MODEL_PREFIX = 'TS'
OUT_FORECAST = f'{MODEL_PREFIX}_delinquency_forecast_US_Q.csv'

# ── File-export toggles ───────────────────────────────────────────────────────
# Both OFF for now: every graph, table and chart stays inline in this notebook.
# Flip to True to also write the LaTeX results table / forecast CSV to disk.
SAVE_LATEX_TABLE = False
SAVE_CSV_OUTPUTS = False

# ── Load the Stage 1 output (GitHub first, local fallback) ─────────────────────
def load_regressors(filename, token=None):
    url = f'{GITHUB_RAW_BASE}/{filename}'
    try:
        if token:
            import requests, io
            r = requests.get(url, headers={'Authorization': f'token {token}'})
            r.raise_for_status()
            out = pd.read_csv(io.StringIO(r.text), index_col=0, parse_dates=True)
        else:
            out = pd.read_csv(url, index_col=0, parse_dates=True)
        print(f'Loaded {filename} from GitHub')
    except Exception as e:
        local = LOCAL_OUTPUT / filename
        out = pd.read_csv(local, index_col=0, parse_dates=True)
        print(f'GitHub load failed ({type(e).__name__}); loaded local copy: {local}')
    return out.asfreq('QE-DEC')

df = load_regressors(INPUT_REGRESSORS, token=GITHUB_TOKEN)

# ── Fail fast on a schema mismatch rather than surfacing as a NaN later ────────
_need = [TARGET] + EXOG + [v for v in VAR_VARS if v != TARGET]
_missing = [c for c in _need if c not in df.columns]
if _missing:
    _avail = [c for c in df.columns if any(c.startswith(m.rsplit('_L', 1)[0]) for m in _missing)]
    raise KeyError(
        f'{len(_missing)} required column(s) absent from {INPUT_REGRESSORS}: {_missing}\n'
        f'  closest available: {_avail}\n'
        f'  upstream lag re-derivations change these names — reconcile EXOG with the file.')

# ── Target provenance ─────────────────────────────────────────────────────────
# Stage1_final_regressors_US_Q.csv supplies us_delinquency_rate ALREADY cubic-spline
# imputed across the COVID window: it equals delinquency_spline for the 10 quarters
# flagged by covid_dummy (2020Q1-2022Q2) and us_delinquency_rate_raw everywhere else.
# This notebook documents its own COVID treatment as NaN-in-place with the SARIMAX
# Kalman filter, so inheriting the upstream spline would (a) double-treat the same
# structural break and (b) feed synthetic values into the backtest as if they were
# observed — the spline is fitted using data either side of the gap, so those values
# embed future information. Default is therefore to take the raw series and apply
# this notebook's own masking.
TARGET_SOURCE = 'raw'          # 'raw' (recommended) | 'adjusted'

_raw_col = f'{TARGET}_raw'
if TARGET_SOURCE == 'raw' and _raw_col in df.columns:
    _n_sub = int((~np.isclose(df[TARGET], df[_raw_col], equal_nan=True)).sum())
    df[TARGET] = df[_raw_col]
    print(f'Target source : {_raw_col} -> {TARGET}  '
          f'({_n_sub} upstream spline-adjusted quarters reverted to raw)')
elif _raw_col in df.columns:
    print(f'Target source : {TARGET} as supplied (upstream COVID spline RETAINED)')
else:
    print(f'Target source : {TARGET} as supplied (no raw column in file)')
print(f'  Shape : {df.shape}')
print(f'  Span  : {df.index.min().date()} -> {df.index.max().date()}')
print(f'  Target: {TARGET} — {df[TARGET].notna().sum()} obs | '
      f'min {df[TARGET].min():.2f}% | max {df[TARGET].max():.2f}% | mean {df[TARGET].mean():.2f}%')

Loaded Stage1_final_regressors_US_Q.csv from GitHub
Target source : us_delinquency_rate_raw -> us_delinquency_rate  (10 upstream spline-adjusted quarters reverted to raw)
  Shape : (164, 43)
  Span  : 1990-03-31 -> 2030-12-31
  Target: us_delinquency_rate — 140 obs | min 1.53% | max 6.77% | mean 3.70%


In [2]:
# ── Split history vs forecast horizon on target availability ──────────────────
def cmask(idx):
    # Union of this notebook's COVID window with any quarter the upstream pipeline
    # flagged as adjusted. Without the union, upstream-imputed quarters falling
    # outside [COVID_START, COVID_END] would enter the training sample and be scored
    # as observations in the backtest.
    idx = pd.DatetimeIndex(idx)
    m = (idx >= COVID_START) & (idx <= COVID_END)
    if 'covid_dummy' in df.columns:
        m = m | idx.isin(df.index[df['covid_dummy'] == 1])
    return m

hist_idx = df.index[df[TARGET].notna()]          # target observed
fc_idx   = df.index[df.index > hist_idx.max()]   # 20 forecast quarters (target NaN)

# Exogenous block starts once all lag columns are populated (early-sample lags NaN).
# Exogenous completeness. Rows are never dropped: SARIMAX requires exog aligned
# one-to-one with endog, and dropping rows would desynchronise them and break the
# quarterly calendar (the same reason COVID is handled NaN-in-place, not by row drop).
#   'trim'        -> start the sample after the last incomplete row; imputes nothing
#   'ffill'       -> carry the last observation forward; uses only past information
#   'interpolate' -> linear fill; NOTE pulls in future values, so avoid for history
EXOG_NAN_POLICY = 'trim'

X_all = df.loc[hist_idx, EXOG]
start = X_all.dropna().index.min()

_Xh  = df.loc[start:hist_idx.max(), EXOG].replace([np.inf, -np.inf], np.nan)
_bad = _Xh.index[~_Xh.notna().all(axis=1)]
if len(_bad):
    print(f'WARNING: {len(_bad)} incomplete exog rows in history '
          f'({_bad.min().date()} .. {_bad.max().date()})  ->  policy = {EXOG_NAN_POLICY}')
    for _c, _n in _Xh.isna().sum().items():
        if _n:
            print(f'    {_c}: {int(_n)} missing')
    if EXOG_NAN_POLICY == 'trim':
        start = _Xh.index[_Xh.index > _bad.max()].min()

y_full = df.loc[start:hist_idx.max(), TARGET]     # observed target, exog-complete window
X_hist = df.loc[start:hist_idx.max(), EXOG].replace([np.inf, -np.inf], np.nan)
if   EXOG_NAN_POLICY == 'ffill':       X_hist = X_hist.ffill().bfill()
elif EXOG_NAN_POLICY == 'interpolate': X_hist = X_hist.interpolate(limit_direction='both')

assert np.isfinite(X_hist.to_numpy(dtype=float)).all(), 'X_hist still contains NaN/inf'
assert X_hist.index.equals(y_full.index), 'exog/endog index mismatch'
X_fc   = df.loc[fc_idx, EXOG].interpolate(limit_direction='both')  # 02a forecast exog

# COVID handling: NaN-in-place for the ARIMA family (state-space handles missing
# natively); a COVID-dropped copy is used only where a method cannot take NaN.
y_nan  = y_full.copy(); y_nan[cmask(y_nan.index)] = np.nan
y_drop = y_full[~cmask(y_full.index)]

# Report any interior NaNs patched in the forecast-period exog (gap-crossing lags).
_patched = df.loc[fc_idx, EXOG].isna().sum()
_patched = _patched[_patched > 0]

print(f'Exog-complete fit window : {start.date()} -> {y_full.index.max().date()}')
print(f'  observations (n)       : {len(y_full)}')
_cm = cmask(y_full.index)
_win = (y_full.index >= COVID_START) & (y_full.index <= COVID_END)
print(f'  COVID quarters -> NaN  : {int(_cm.sum())}  '
      f'(window {COVID_START} .. {COVID_END}'
      + (f' + {int((_cm & ~_win).sum())} upstream-flagged beyond it'
         if int((_cm & ~_win).sum()) else '') + ')')
if int((_cm & ~_win).sum()):
    print('    also masked: '
          + ', '.join(str(d.date()) for d in y_full.index[_cm & ~_win]))
print(f'  usable after COVID drop: {len(y_drop)}')
print(f'Forecast horizon         : {fc_idx.min().date()} -> {fc_idx.max().date()}  '
      f'({len(fc_idx)} quarters)')
if len(_patched):
    print('  forecast-exog NaNs patched by interpolation:')
    for k, v in _patched.items(): print(f'    {k}: {int(v)}')
else:
    print('  forecast-exog: no NaNs')

# ── Collinearity audit on the exogenous block (VIF + condition number) ─────────
Xv = X_hist.loc[y_drop.index].dropna()
Xc = np.column_stack([np.ones(len(Xv)), Xv.values])
vif = pd.Series({c: variance_inflation_factor(Xc, i + 1) for i, c in enumerate(Xv.columns)})
cond = np.linalg.cond(Xv.values - Xv.values.mean(0))
print('\nExogenous-block collinearity (VIF; rule-of-thumb concern > 5):')
for c, v in vif.sort_values(ascending=False).items():
    print(f'  {c:<28s} VIF={v:5.2f}   corr(target)={y_drop.reindex(Xv.index).corr(Xv[c]):+.2f}')
print(f'  condition number (centred) = {cond:.1f}')

Exog-complete fit window : 1991-12-31 -> 2025-12-31
  observations (n)       : 137
  COVID quarters -> NaN  : 10  (window 2020-03-31 .. 2021-12-31 + 2 upstream-flagged beyond it)
    also masked: 2022-03-31, 2022-06-30
  usable after COVID drop: 127
Forecast horizon         : 2026-03-31 -> 2030-12-31  (20 quarters)
  forecast-exog: no NaNs

Exogenous-block collinearity (VIF; rule-of-thumb concern > 5):
  us_unemployment_L0           VIF= 3.49   corr(target)=+0.39
  us_house_price_yoy_L3        VIF= 2.46   corr(target)=-0.46
  us_gdp_yoy_growth_L2         VIF= 2.40   corr(target)=-0.25
  us_consumer_confidence_L2    VIF= 1.95   corr(target)=-0.32
  us_credit_qoq_growth_L6      VIF= 1.51   corr(target)=+0.35
  us_cpi_L6                    VIF= 1.27   corr(target)=+0.17
  condition number (centred) = 7.4


## **1: Sample Diagnostics & EDA — the COVID transmission break**
___
The single most important modelling fact for this target: **the pandemic severed the
macro→credit link.** In every prior recession, rising unemployment pulled delinquency
up (the GFC took it to ~6.8%). During COVID, mass forbearance and stimulus pushed
delinquency *down* to ~1.5% even as unemployment spiked to 11%. A macro-conditional
model fit through that window would learn a relationship that never held structurally —
so those quarters are excluded from fitting and from evaluation.

In [3]:
# Figure 1 — delinquency vs unemployment, with the COVID inversion highlighted
unemp = df['us_unemployment'] if 'us_unemployment' in df.columns else None
fig = make_subplots(specs=[[{'secondary_y': True}]])

for s, e, lbl in RECESSIONS_US:
    fig.add_vrect(x0=s, x1=e, fillcolor='lightgrey', opacity=0.3, layer='below', line_width=0)
fig.add_vrect(x0=COVID_START, x1=COVID_END, fillcolor=AMBER, opacity=0.12,
              layer='below', line_width=0,
              annotation_text='COVID excluded', annotation_position='top left',
              annotation_font_size=10)

fig.add_trace(go.Scatter(x=y_full.index, y=y_full.values, mode='lines',
                         line=dict(color=NAVY, width=1.8), name='Delinquency rate (%)'),
              secondary_y=False)
if unemp is not None:
    um = unemp.loc[start:hist_idx.max()]
    fig.add_trace(go.Scatter(x=um.index, y=um.values, mode='lines',
                             line=dict(color=RED, width=1.4, dash='dot'), name='Unemployment (%)'),
                  secondary_y=True)

fig.update_layout(
    title=dict(text='Figure 1 — Delinquency vs unemployment: the COVID inversion<br>'
                    '<sup>Grey = NBER recessions | amber = COVID window excluded from fit & '
                    'evaluation | note the divergence in 2020–21</sup>',
               font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=460,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1, font=dict(size=10)),
    margin=dict(t=95, b=40, l=55, r=55))
fig.update_yaxes(title_text='Delinquency (%)', secondary_y=False, color=NAVY)
fig.update_yaxes(title_text='Unemployment (%)', secondary_y=True, color=RED)
fig.show()

## **2: Model Specifications**
___
Orders are selected **once** (via `auto_arima` on the COVID-dropped target) and then
only re-estimated during the backtest — the same specification-once convention as 02a,
noted as a mild source of optimism in the caveats. The DM/Clark-West helpers are the
**same functions used in 02a** (Harvey-Leybourne-Newbold small-sample correction; the
Clark-West nested-model adjustment), reproduced here so this notebook is self-contained.

**AR(1) — requested baseline**

$$y_t = \phi_0 + \phi_1 y_{t-1} + \varepsilon_t$$

A first-order autoregression on the level, fit as SARIMAX(1,0,0) with no differencing
and no exogenous input. Included because it was specifically requested as a baseline,
not because it is expected to compete (Box & Jenkins 1976; Hamilton 1994, Ch. 3).

**SARIMA — best univariate ARIMA**

$$\Phi_P(L^4)\,\phi_p(L)\,(1-L)^d(1-L^4)^D\,y_t = \Theta_Q(L^4)\,\theta_q(L)\,\varepsilon_t$$

Order $(p,d,q)\times(P,D,Q)_4$ chosen **once** by AIC via `auto_arima` on the
COVID-dropped series (Hyndman & Khandakar 2008); functional form per Box & Jenkins
(1976). Carries no exogenous information — the pure-time-series counterpart to
SARIMAX/ARDL below.

**SARIMAX — macro-conditional, single-equation**

$$\Phi_P(L^4)\,\phi_p(L)\,(1-L)^d(1-L^4)^D\,y_t = \boldsymbol{\beta}^\top\mathbf{x}_t + \Theta_Q(L^4)\,\theta_q(L)\,\varepsilon_t$$

Same SARIMA backbone plus the six CCF-lagged regressors $\mathbf{x}_t$ (identical
block to the Stage 2 OLS regressors) entered as exogenous inputs. Order selected
once by AIC. Because it takes the other variables as given, this is one of the two
models that genuinely consumes 02a's forecasted regressors beyond $h=1$.

**ARDL($p$) — hand-rolled autoregressive distributed lag**

$$y_t = \beta_0 + \sum_{i=1}^{p}\gamma_i\,y_{t-i} + \boldsymbol{\beta}^\top\mathbf{x}_t + \varepsilon_t$$

Estimated by OLS via a calendar-correct lag matrix — hand-rolled rather than a
packaged implementation, so the design matrix is unambiguous and auditable. Lag
order $p$ chosen once by AIC over $p \in \{1,\dots,4\}$. **Note for the record:**
this is a static dynamic-regression ARDL, not a Pesaran-Shin-Smith bounds-testing /
cointegrating ARDL — no cointegration test is run and none is implied by the name here.

**VAR($k$) — joint system, independent cross-check**

$$\mathbf{v}_t = \mathbf{c} + \sum_{i=1}^{k}\mathbf{A}_i\,\mathbf{v}_{t-i} + \mathbf{u}_t$$

Fit on the five-variable contemporaneous block $\mathbf{v}_t$ (target + unemployment,
GDP growth, CPI, credit growth), differenced once if an ADF test (Dickey & Fuller
1979) fails to reject a unit root in any component series; lag $k$ chosen once by
AIC (Hamilton 1994, Ch. 11). Forecasts all variables jointly from their own
dynamics, so — per the note in Cell 0 — it does **not** consume 02a's forecasts and
serves purely as an independent cross-check.

In [4]:
# ── Model-class label (verbatim from 02a) ─────────────────────────────────────
def classify_model(order, seasonal_order):
    p, d, q = order; P, D, Q, m = seasonal_order
    if p == 0 and q == 0 and P == 0 and Q == 0: return 'Mean'
    if P > 0 or Q > 0 or D > 0:                  return 'SARMA'
    if q == 0 and Q == 0:                        return 'AR'
    return 'ARMA'

# ── Diebold-Mariano with HLN small-sample correction (HAC when h>1) — from 02a ──
def dm_hln(e_bench, e_model, h=BACKTEST_H):
    d = e_bench ** 2 - e_model ** 2; n = len(d)      # >0 => model beats benchmark
    if n < 3: return np.nan, np.nan
    var = np.var(d, ddof=0)
    for k in range(1, h):
        var += 2 * np.cov(d[:-k], d[k:])[0, 1]
    var /= n
    if var <= 0: return np.nan, np.nan               # negative LRV -> DM undefined
    dm   = d.mean() / np.sqrt(var)
    corr = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)   # Harvey-Leybourne-Newbold 1997
    dm_s = dm * corr
    return dm_s, 2 * (1 - stats.t.cdf(abs(dm_s), df=n - 1))

# ── Clark-West adjustment (correct test when the benchmark is NESTED) — from 02a ─
def clark_west(e_bench, e_model, f_bench, f_model):
    ft = e_bench ** 2 - (e_model ** 2 - (f_bench - f_model) ** 2); n = len(ft)
    if n < 3: return np.nan, np.nan
    t = ft.mean() / (ft.std(ddof=1) / np.sqrt(n))
    return t, 1 - stats.norm.cdf(t)                  # one-sided: model better

# ── Hand-rolled ARDL: calendar-correct lag matrix, OLS, recursive forecast ─────
def ardl_design(y, X, p):
    d = pd.DataFrame({'y': y})
    for i in range(1, p + 1): d[f'yL{i}'] = y.shift(i)
    for c in X.columns:       d[c] = X[c]
    d = d.dropna()
    M_ = np.column_stack([np.ones(len(d))]
                         + [d[f'yL{i}'].values for i in range(1, p + 1)]
                         + [d[c].values for c in X.columns])
    return d.index, d['y'].values, M_

def ardl_fit(y, X, p):
    _, Y, Mx = ardl_design(y, X, p)
    b, *_ = np.linalg.lstsq(Mx, Y, rcond=None)
    return b, Y - Mx @ b

def ardl_pick_p(y, X, pmax=4):
    best = (np.inf, 1)
    for p in range(1, pmax + 1):
        _, Y, Mx = ardl_design(y, X, p)
        b, *_ = np.linalg.lstsq(Mx, Y, rcond=None)
        r = Y - Mx @ b; n = len(Y); k = Mx.shape[1]
        aic = n * np.log(np.sum(r ** 2) / n) + 2 * k
        if aic < best[0]: best = (aic, p)
    return best[1]

def ardl_forecast(b, y_obs, Xf, p):
    buf = list(y_obs.dropna().values); cols = list(Xf.columns); out = []
    for h in range(len(Xf)):
        yl = [buf[-i] for i in range(1, p + 1)]
        yhat = b[0] + sum(b[i] * yl[i - 1] for i in range(1, p + 1)) + b[1 + p:] @ Xf.iloc[h][cols].values
        out.append(yhat); buf.append(yhat)
    return np.array(out)

# ── Select orders ONCE ────────────────────────────────────────────────────────
sar = auto_arima(y_drop, seasonal=True, m=M, information_criterion='aic', stepwise=True,
                 error_action='ignore', suppress_warnings=True,
                 max_p=3, max_q=3, max_P=2, max_Q=2)
Xd = X_hist.loc[y_drop.index]
sarx = auto_arima(y_drop, exogenous=Xd.values, seasonal=True, m=M, information_criterion='aic',
                  stepwise=True, error_action='ignore', suppress_warnings=True,
                  max_p=3, max_q=3, max_P=1, max_Q=1)
SARIMA_ORD,  SARIMA_SORD  = sar.order,  sar.seasonal_order
SARIMAX_ORD, SARIMAX_SORD = sarx.order, sarx.seasonal_order
ARDL_P = ardl_pick_p(y_nan, X_hist)

# ── VAR differencing decision + lag (fixed ONCE for reuse) ─────────────────────
Vfull = df.loc[start:hist_idx.max(), VAR_VARS]
V0 = Vfull.dropna(); V0 = V0[~cmask(V0.index)]
VAR_DIFF = any((adfuller(V0[c].dropna(), autolag='AIC')[1] >= 0.05) for c in V0.columns)
Vm0 = V0.diff().dropna() if VAR_DIFF else V0
VAR_LAG = VAR(Vm0).fit(maxlags=4, ic='aic').k_ar

print('Selected specifications')
print(f'  AR(1)   : (1,0,0)')
print(f'  SARIMA  : {SARIMA_ORD} x {SARIMA_SORD}  [{classify_model(SARIMA_ORD, SARIMA_SORD)}]')
print(f'  SARIMAX : {SARIMAX_ORD} x {SARIMAX_SORD} + {len(EXOG)} exog')
print(f'  ARDL    : p={ARDL_P}  + {len(EXOG)} exog')
print(f'  VAR     : {len(VAR_VARS)} vars | differenced={VAR_DIFF} | lag={VAR_LAG}')

Selected specifications
  AR(1)   : (1,0,0)
  SARIMA  : (1, 1, 1) x (2, 0, 1, 4)  [SARMA]
  SARIMAX : (2, 1, 2) x (1, 0, 1, 4) + 6 exog
  ARDL    : p=1  + 6 exog
  VAR     : 5 vars | differenced=True | lag=4


## **3: Out-of-Sample Backtest & Diebold-Mariano**
___
RMSE and the Diebold-Mariano test are *out-of-sample* quantities, so we run a
**recursive (expanding-window) rolling-origin backtest**: at each origin the selected
spec is re-estimated on data up to that point, a one-step forecast is made against the
naive random walk, and the error is recorded (Diebold & Mariano 1995).

**Design (mirrors 02a).**
- *Scheme:* expanding window — maximises training data at this sample size.
- *Horizon:* `h = 1`. One-step loss differentials are serially uncorrelated (cleanest DM);
  crucially, at `h=1` every lagged regressor is already realised, so the conditional models
  are tested fairly with **no** forecasted inputs.
- *Benchmark:* naive random walk.
- *`beats_RW` flag:* a random walk is nested in the differenced ARIMA specs, where the
  standard DM test is invalid — so the flag uses the **Clark-West** nested-model test and
  additionally requires the model to beat naive in realised RMSE, so it never contradicts
  the RMSE column. The HLN-DM statistic is reported alongside for transparency.
- Origins whose target lands in the COVID window are skipped.

A focused **HLN-DM** then compares the best macro-conditional model against the best
pure-time-series model (a non-nested, two-sided test) to answer directly: *does adding
the macro block beat a good univariate model out-of-sample?*

In [5]:
# ── One-step forecast from a training window (NaN-in-place already applied) ────
def refit_forecast(name, tr_y, tr_X, fut_X):
    if name == 'Naive RW':
        return float(tr_y.dropna().iloc[-1])
    if name == 'AR(1)':
        m = SARIMAX(tr_y, order=(1, 0, 0), enforce_stationarity=False,
                    enforce_invertibility=False).fit(disp=False, maxiter=100)
        return float(m.forecast(1).iloc[-1])
    if name == 'SARIMA':
        m = SARIMAX(tr_y, order=SARIMA_ORD, seasonal_order=SARIMA_SORD,
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100)
        return float(m.forecast(1).iloc[-1])
    if name == 'SARIMAX':
        m = SARIMAX(tr_y, exog=tr_X, order=SARIMAX_ORD, seasonal_order=SARIMAX_SORD,
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100)
        return float(m.get_forecast(1, exog=fut_X).predicted_mean.iloc[-1])
    if name == 'ARDL':
        bb, _ = ardl_fit(tr_y, tr_X, ARDL_P)
        return float(ardl_forecast(bb, tr_y, fut_X, ARDL_P)[0])
    if name == 'VAR':
        Vt = tr_X                                   # tr_X carries the VAR block here
        Vmm = Vt.diff().dropna() if VAR_DIFF else Vt
        r = VAR(Vmm).fit(maxlags=4, ic='aic')
        f = pd.DataFrame(r.forecast(Vmm.values[-r.k_ar:], steps=1), columns=Vmm.columns)[TARGET].values[0]
        return float(Vt[TARGET].iloc[-1] + f) if VAR_DIFF else float(f)

# ── Recursive rolling-origin loop ─────────────────────────────────────────────
MODELS = ['Naive RW', 'AR(1)', 'SARIMA', 'SARIMAX', 'ARDL', 'VAR']
qi = y_nan.index
err = {m: [] for m in MODELS}; fcv = {m: [] for m in MODELS}; act = []; dts = []
n_used = 0
for t in range(BACKTEST_MIN_TRAIN, len(y_nan) - BACKTEST_H + 1):
    tgt = t + BACKTEST_H - 1
    if (qi[tgt].to_period('Q') - qi[t - 1].to_period('Q')).n != BACKTEST_H:   # gap guard
        continue
    if cmask([qi[tgt]])[0]:                                                    # target in COVID
        continue
    a = y_nan.iloc[tgt]
    if not np.isfinite(a):
        continue
    tr_y = y_nan.iloc[:t]; tr_X = X_hist.iloc[:t]; fut_X = X_hist.iloc[[tgt]]
    tr_V = Vfull.iloc[:t].dropna()
    try:
        preds = {}
        for m in MODELS:
            if   m == 'VAR':                     preds[m] = refit_forecast(m, tr_y, tr_V, None)
            elif m in ('SARIMAX', 'ARDL'):       preds[m] = refit_forecast(m, tr_y, tr_X, fut_X)
            else:                                preds[m] = refit_forecast(m, tr_y, None, None)
    except Exception:
        continue
    if not all(np.isfinite(list(preds.values()))):
        continue
    for m in MODELS:
        err[m].append(a - preds[m]); fcv[m].append(preds[m])
    act.append(a); dts.append(qi[tgt]); n_used += 1
print(f'Recursive expanding-window backtest | h={BACKTEST_H} | benchmark = naive RW')
print(f'Origins used: {n_used}  ({pd.Timestamp(dts[0]).date()} -> {pd.Timestamp(dts[-1]).date()})\n')

# ── Error metrics (MSE / RMSE / MAE / MAPE) + skill vs naive RW ────────────────
# Delinquency is strictly positive (min ~1.5%), so MAPE is well-defined here.
a_arr = np.array(act)
en = np.array(err['Naive RW']); fn = np.array(fcv['Naive RW'])
rmse_n = np.sqrt(np.mean(en ** 2))

res_rows, diag_rows = [], []
for m in MODELS:
    em = np.array(err[m]); fm = np.array(fcv[m])
    mse  = float(np.mean(em ** 2))
    rmse = float(np.sqrt(mse))
    mae  = float(np.mean(np.abs(em)))
    mape = float(np.mean(np.abs(em / a_arr)) * 100)
    if m == 'Naive RW':
        res_rows.append({'model': m, 'MSE': round(mse, 4), 'RMSE': round(rmse, 4),
                         'MAE': round(mae, 4), 'MAPE%': round(mape, 2),
                         'skill%': 0.0, 'beats_RW': '—'})
        diag_rows.append({'model': m, 'DM*': None, 'DM_p': None, 'CW': None, 'CW_p': None})
        continue
    skill = 100 * (1 - rmse / rmse_n)
    dm, pdm = dm_hln(en, em, BACKTEST_H); cw, pcw = clark_west(en, em, fn, fm)
    beats = (rmse < rmse_n) and np.isfinite(pcw) and pcw < 0.05
    res_rows.append({'model': m, 'MSE': round(mse, 4), 'RMSE': round(rmse, 4),
                     'MAE': round(mae, 4), 'MAPE%': round(mape, 2),
                     'skill%': round(skill, 1), 'beats_RW': 'Yes' if beats else 'No'})
    diag_rows.append({'model': m,
                      'DM*': round(dm, 3) if np.isfinite(dm) else None,
                      'DM_p': round(pdm, 4) if np.isfinite(pdm) else None,
                      'CW': round(cw, 3) if np.isfinite(cw) else None,
                      'CW_p': round(pcw, 4) if np.isfinite(pcw) else None})

results = pd.DataFrame(res_rows).set_index('model').sort_values('RMSE')
diag    = pd.DataFrame(diag_rows).set_index('model').loc[results.index]

# ── Results table (headline) — error metrics + skill, best first ───────────────
print('Results (MSE/RMSE/MAE in native % units; MAPE scale-free; skill vs naive RW):')
display(results)

# ── Diagnostics table (significance tests) — mirrors 02a's split ───────────────
print('Diagnostics — DM* = HLN-corrected Diebold-Mariano (Harvey-Leybourne-Newbold), '
      'CW = Clark-West nested-benchmark test (one-sided); both >0 favour the model, '
      'p < 0.05 = significant:')
display(diag)

# ── Focused non-nested test: best macro-conditional vs best pure-TS ────────────
ts_pool    = {m: np.sqrt(np.mean(np.array(err[m]) ** 2)) for m in ['AR(1)', 'SARIMA']}
macro_pool = {m: np.sqrt(np.mean(np.array(err[m]) ** 2)) for m in ['SARIMAX', 'ARDL']}
best_ts    = min(ts_pool,    key=ts_pool.get)
best_macro = min(macro_pool, key=macro_pool.get)
dm_h, p_h  = dm_hln(np.array(err[best_ts]), np.array(err[best_macro]), BACKTEST_H)
print(f'Does macro beat pure time-series?  {best_macro} vs {best_ts} '
      f'(HLN-DM, two-sided, non-nested):')
print(f'  DM* = {dm_h:+.3f} | p = {p_h:.4f}  ->  '
      f'{"macro significantly better" if (p_h < 0.05 and dm_h > 0) else "no significant difference at 5%"}')

# ── booktabs LaTeX export of the results table (mirrors 02a) ───────────────────
def _tex_escape(s): return str(s).replace('_', r'\_')
def report_to_latex(df):
    out = [r'\begin{tabular}{lrrrrrc}', r'  \toprule',
           r'  Model & MSE & RMSE & MAE & MAPE\% & Skill\% & Beats RW \\', r'  \midrule']
    for m, r in df.iterrows():
        out.append('  {} & {} & {} & {} & {} & {} & {} \\\\'.format(
            _tex_escape(m), r['MSE'], r['RMSE'], r['MAE'], r['MAPE%'],
            '--' if pd.isna(r['skill%']) else r['skill%'], r['beats_RW']))
    out += [r'  \bottomrule', r'\end{tabular}']
    return '\n'.join(out)

latex_table = report_to_latex(results)
if SAVE_LATEX_TABLE:
    try:
        (LOCAL_OUTPUT / f'{MODEL_PREFIX}_delinquency_backtest_US.tex').write_text(latex_table)
        print(f'LaTeX results table -> {LOCAL_OUTPUT / (MODEL_PREFIX + "_delinquency_backtest_US.tex")}')
    except Exception as e:
        print(f'(LaTeX not written to disk: {e})')
else:
    print('LaTeX results table (disk export OFF — set SAVE_LATEX_TABLE=True to re-enable):')
print(latex_table)

Recursive expanding-window backtest | h=1 | benchmark = naive RW
Origins used: 67  (2006-12-31 -> 2025-12-31)

Results (MSE/RMSE/MAE in native % units; MAPE scale-free; skill vs naive RW):


,MSE,RMSE,MAE,MAPE%,skill%,beats_RW
model,,,,,,
ARDL,0.0369,0.1922,0.1293,3.80,22.1,Yes
SARIMA,0.0460,0.2145,0.1405,3.97,13.1,Yes
VAR,0.0521,0.2283,0.1385,3.88,7.5,Yes
AR(1),0.0588,0.2425,0.1549,4.17,1.7,No
SARIMAX,0.0604,0.2457,0.1470,3.95,0.4,Yes
Naive RW,0.0609,0.2467,0.1567,4.28,0.0,—


Diagnostics — DM* = HLN-corrected Diebold-Mariano (Harvey-Leybourne-Newbold), CW = Clark-West nested-benchmark test (one-sided); both >0 favour the model, p < 0.05 = significant:


,DM*,DM_p,CW,CW_p
model,,,,
ARDL,1.992,0.0505,3.364,0.0004
SARIMA,1.621,0.1099,2.894,0.0019
VAR,0.543,0.5887,2.079,0.0188
AR(1),0.719,0.4749,0.880,0.1894
SARIMAX,0.052,0.9588,2.001,0.0227
Naive RW,NaN,NaN,NaN,NaN


Does macro beat pure time-series?  ARDL vs SARIMA (HLN-DM, two-sided, non-nested):
  DM* = +1.095 | p = 0.2775  ->  no significant difference at 5%
LaTeX results table (disk export OFF — set SAVE_LATEX_TABLE=True to re-enable):
\begin{tabular}{lrrrrrc}
  \toprule
  Model & MSE & RMSE & MAE & MAPE\% & Skill\% & Beats RW \\
  \midrule
  ARDL & 0.0369 & 0.1922 & 0.1293 & 3.8 & 22.1 & Yes \\
  SARIMA & 0.046 & 0.2145 & 0.1405 & 3.97 & 13.1 & Yes \\
  VAR & 0.0521 & 0.2283 & 0.1385 & 3.88 & 7.5 & Yes \\
  AR(1) & 0.0588 & 0.2425 & 0.1549 & 4.17 & 1.7 & No \\
  SARIMAX & 0.0604 & 0.2457 & 0.147 & 3.95 & 0.4 & Yes \\
  Naive RW & 0.0609 & 0.2467 & 0.1567 & 4.28 & 0.0 & — \\
  \bottomrule
\end{tabular}


In [6]:
# Figure 2 — one-step OOS predictions vs actual (all models)
oos = pd.DataFrame({'actual': act}, index=pd.DatetimeIndex(dts))
for m in MODELS: oos[m] = fcv[m]
colmap = {'Naive RW': GREY, 'AR(1)': AMBER, 'SARIMA': GREEN,
          'SARIMAX': BLUE, 'ARDL': RED, 'VAR': '#8E44AD'}

fig = go.Figure()
fig.add_trace(go.Scatter(x=oos.index, y=oos['actual'], mode='lines',
                         line=dict(color=NAVY, width=2.4), name='Actual'))
for m in MODELS:
    fig.add_trace(go.Scatter(x=oos.index, y=oos[m], mode='lines',
                             line=dict(color=colmap[m], width=1.2,
                                       dash='dot' if m == 'Naive RW' else 'solid'),
                             opacity=0.9, name=m))
fig.update_layout(
    title=dict(text='Figure 2 — One-step out-of-sample predictions vs actual<br>'
                    '<sup>Expanding-window recursive backtest | COVID targets excluded</sup>',
               font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=460, yaxis_title='Delinquency rate (%)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1, font=dict(size=10)),
    margin=dict(t=95, b=40, l=55, r=20))
fig.show()

### Out-of-Sample Model Comparison

**Error metrics** (h = 1, expanding-window backtest, 67 origins, 2006 Q4 – 2025 Q4,
COVID-adjusted quarters excluded):

| Model | MSE | RMSE | MAE | MAPE% | Skill vs RW (%) | Beats RW |
|---|---|---|---|---|---|---|
| ARDL | 0.0369 | 0.1922 | 0.1293 | 3.80 | 22.1 | Yes |
| SARIMA | 0.0474 | 0.2176 | 0.1410 | 3.99 | 11.8 | Yes |
| VAR | 0.0521 | 0.2283 | 0.1385 | 3.88 | 7.5 | Yes |
| AR(1) | 0.0588 | 0.2425 | 0.1549 | 4.17 | 1.7 | No |
| SARIMAX | 0.0602 | 0.2454 | 0.1484 | 3.98 | 0.5 | Yes |
| Naive RW | 0.0609 | 0.2467 | 0.1567 | 4.28 | 0.0 | — |

ARDL ranks first on **all four** error metrics — 22.1% root-mean-squared-error skill over
the naive random walk — reflecting the value of the six lagged macro regressors at a
horizon where every lag is already realised. Unlike the earlier configuration there is
no metric on which a different model wins, so the ranking is unambiguous.

**SARIMAX is the notable casualty of this configuration**, falling to 0.5% skill and
fifth place. Two things drove it: the order search returned a much richer specification
here, (2,1,2)×(1,0,1,4), and the target now retains the true COVID trough as masked
missing data rather than inheriting an upstream spline, leaving two fewer usable
observations. On realised squared error it is now indistinguishable from the naive
benchmark.

**Significance** (DM\* = HLN-corrected Diebold-Mariano; CW = Clark-West nested-benchmark
test, one-sided; both >0 favour the model, p < 0.05 = significant):

| Model | DM\* | DM p | CW | CW p |
|---|---|---|---|---|
| ARDL | 1.992 | 0.0505 | 3.364 | 0.0004 |
| SARIMA | 1.499 | 0.1387 | 2.789 | 0.0026 |
| VAR | 0.543 | 0.5887 | 2.079 | 0.0188 |
| AR(1) | 0.719 | 0.4749 | 0.880 | 0.1894 |
| SARIMAX | 0.067 | 0.9467 | 1.969 | 0.0245 |

Because the naive random walk is nested inside the differenced SARIMA/SARIMAX/ARDL
specifications, the standard DM test is not valid there — the `beats_RW` column is
decided by Clark-West, which is valid under nesting, not by DM\* alone. Note that
ARDL's DM p-value is 0.0505, fractionally *above* the 5% threshold; on the non-nested
statistic it is borderline rather than significant, and should be reported as such.

**SARIMAX illustrates why the nested/non-nested distinction matters.** Its DM\* of 0.067
(p = 0.9467) says it is no better than the naive random walk on realised error, yet
Clark-West returns p = 0.0245. These are not contradictory: Clark-West corrects for the
fact that a nested larger model must estimate extra parameters that are zero under the
null, and so tests whether the *population* model has predictive content rather than
whether the estimated model beat the benchmark in this sample. Read together they say
SARIMAX's exogenous block carries genuine signal that its realised forecasts failed to
convert at this sample size. AR(1) fails both tests (CW p = 0.1894) and is retained only
because it was the requested baseline.

**Does the macro block add anything over a good univariate model?** A direct,
non-nested HLN-DM test between the best macro-conditional model (ARDL) and the best pure
time-series model (SARIMA) gives DM\* = +1.081, p = 0.2836 — **no significant difference
at the 5% level**. So while ARDL sweeps the point-error metrics, its advantage over
SARIMA is not statistically distinguishable from noise at this sample size (n = 67
origins). This notebook does not pick a single winner: all six forward paths are carried
into Section 4 and retained for comparison against the machine-learning PD benchmark.

## **3b: Specification-Search Robustness — Is the Pre-Specified Block Defensible?**

The exogenous block `EXOG` is fixed *a priori*: six predictors at their
cross-correlation-function (CCF) optimal lags, taken from EDA Section 5 and held
deliberately identical to the Stage 2 regressor set so that this notebook remains a
like-for-like benchmark of the Stage 2 probability-of-default (PD) model. That choice
invites an obvious challenge — **"you never tested the alternatives"** — and this
section answers it directly.

Only the exogenous block varies here. The autoregressive distributed-lag (ARDL) order
is held at `ARDL_P` and the one-step-ahead expanding-window origin scheme is unchanged,
so every difference in root-mean-squared error (RMSE) is attributable to the regressor
set alone.

**Three selection regimes, in increasing order of honesty:**

1. **Subset chosen by realised out-of-sample RMSE** on the full sample. This optimises
   the very metric being reported, and is therefore maximally contaminated. It is
   included precisely to show how large a number this procedure can produce.
2. **Subset chosen once by in-sample Akaike information criterion (AIC)** on the full
   sample — both exhaustively and by forward stepwise selection. This is the *same*
   regime `auto_arima` uses to fix the SARIMA and SARIMAX orders in Section 2.
3. **Subset re-chosen by AIC inside every training window.** No forecast origin sees
   its own future, so this is the only regime whose realised accuracy is reproducible
   in production.

**Two placebos separate genuine signal from search artefact:**

- **Placebo A (replacement, primary).** All six regressors are swapped for an *equal
  number* of pure standard-normal draws and the identical 63-subset search is repeated.
  Equal search space, zero true signal by construction — so whatever gain appears here
  is pure artefact, and gives a benchmark against which the real gain can be judged.
- **Placebo B (contamination, secondary).** Noise regressors are added *alongside* the
  real ones, and we record how often noise enters the top-ranked subsets. Note that the
  RMSE comparison in B is **not** evidence: adding regressors makes the search space a
  superset, so the best subset can only improve mechanically. Only the
  selection-frequency statistic from B is informative.

**One caveat on sample.** This section requires every candidate regressor to be
non-missing simultaneously, which yields a shorter common sample than Section 3's
headline backtest. Figures here are internally comparable but **must not** be compared
against the Section 3 table.

In [7]:
# ── Section 3b: does the pre-specified exogenous block cost us anything? ───────
# The EXOG block is fixed a priori (cross-correlation-function-optimal lags from
# EDA Section 5, identical to the Stage 2 regressor set). A fair examiner challenge
# is: "you never tested the alternatives." This section tests them, and separates
# how much of any apparent improvement is real from how much is a search artefact.
#
# Design: only the exogenous block varies. Autoregressive distributed-lag (ARDL)
# lag order is held at ARDL_P and the h=1 expanding-window origin scheme is
# unchanged, so every RMSE difference is attributable to the regressor set alone.
#
# Three selection regimes, in increasing order of honesty:
#   [1] subset chosen by realised out-of-sample RMSE on the full sample -> optimises
#       the very metric being reported; maximally contaminated
#   [2] subset chosen once by in-sample Akaike information criterion (AIC) on the
#       full sample -> the same regime auto_arima uses for the SARIMA/SARIMAX orders
#   [3] subset re-chosen by AIC inside EVERY training window -> no look-ahead; this
#       is the only regime whose realised accuracy is achievable in production
#
# Two placebos quantify how much of regime [1]'s gain is manufactured by searching:
#   [A] REPLACEMENT (primary): swap all real regressors for an EQUAL number of pure
#       standard-normal regressors and repeat the identical 63-subset search. Equal
#       search space, zero true signal -> the gain measured here is pure artefact.
#   [B] CONTAMINATION (secondary): add noise regressors alongside the real ones and
#       record how often noise enters the winning subset. NOTE: the RMSE comparison
#       in [B] is NOT evidence, because adding regressors makes the search space a
#       superset and the best subset can only improve mechanically. Only the
#       selection-frequency statistic from [B] is informative.
#
# NOTE: this section uses a COMMON sample across all candidate regressors (every
# candidate must be non-NaN), which is a shorter window than Section 3's headline
# backtest. RMSEs here are internally comparable but NOT comparable to Section 3.

import re
from itertools import combinations
from collections import Counter

RUN_SEARCH_ROBUSTNESS = True     # set False to skip (~1 min runtime)
SR_SEEDS_A            = 10       # replacement-placebo repetitions (primary)
SR_SEEDS_B            = 5        # contamination-placebo repetitions (secondary)
SR_N_NOISE            = 3        # noise regressors added in placebo [B]

if RUN_SEARCH_ROBUSTNESS:

    # ── Candidate pool: every lagged regressor the pipeline provides ───────────
    sr_extra = [c for c in df.columns
                if re.search(r'_L\d+$', c) and c not in EXOG and TARGET not in c]
    SR_POOL = list(EXOG) + sr_extra
    print(f'Candidate pool: {len(SR_POOL)} lagged regressors '
          f'({len(EXOG)} pre-specified, {len(sr_extra)} additional)')
    for c in sr_extra:
        print(f'    + {c}')

    # ── Common sample so every subset is scored on identical quarters ──────────
    sr_work = pd.DataFrame({'y': y_nan, 'yL1': y_nan.shift(1)}).join(df[SR_POOL])
    sr_work = sr_work[sr_work.notna().all(axis=1)]
    sr_dates = sr_work.index
    sr_y     = sr_work['y'].values
    sr_ylag  = sr_work['yL1'].values
    sr_X     = sr_work[SR_POOL].values
    sr_cix   = {c: i for i, c in enumerate(SR_POOL)}
    print(f'\nCommon estimation sample : {sr_dates.min().date()} -> {sr_dates.max().date()}  '
          f'(n={len(sr_work)})')
    print(f'  Section 3 headline was : {y_nan.index.min().date()} -> {y_nan.index.max().date()}  '
          f'(n={len(y_nan)})   <- RMSEs NOT cross-comparable')

    sr_contig = np.array([False] + [
        (sr_dates[t].to_period('Q') - sr_dates[t - 1].to_period('Q')).n == 1
        for t in range(1, len(sr_dates))])
    SR_ORIGINS = [t for t in range(BACKTEST_MIN_TRAIN, len(sr_dates))
                  if sr_contig[t] and not cmask([sr_dates[t]])[0]]
    print(f'  origins                : {len(SR_ORIGINS)}  '
          f'({sr_dates[SR_ORIGINS[0]].date()} -> {sr_dates[SR_ORIGINS[-1]].date()})')

    # ── Fast ARDL path, cross-checked against the notebook's own ardl_fit ─────
    def sr_design(t, cix, Xm):
        return np.column_stack([np.ones(t), sr_ylag[:t], Xm[:t][:, cix]])

    def sr_aic(t, cix, Xm):
        M = sr_design(t, cix, Xm)
        if t <= M.shape[1] + 2:
            return np.inf
        b, *_ = np.linalg.lstsq(M, sr_y[:t], rcond=None)
        r = sr_y[:t] - M @ b
        return t * np.log(np.sum(r ** 2) / t) + 2 * M.shape[1]

    def sr_backtest(cix, Xm=None):
        Xm = sr_X if Xm is None else Xm
        e = np.empty(len(SR_ORIGINS))
        for j, t in enumerate(SR_ORIGINS):
            M = sr_design(t, cix, Xm)
            b, *_ = np.linalg.lstsq(M, sr_y[:t], rcond=None)
            e[j] = sr_y[t] - np.concatenate([[1.0, sr_ylag[t]], Xm[t, cix]]) @ b
        return e

    sr_rmse = lambda e: float(np.sqrt(np.mean(e ** 2)))

    def sr_stepwise(t, pool_cix, Xm):
        """Greedy forward selection by AIC using data up to t only."""
        cur, best = [], sr_aic(t, [], Xm)
        while True:
            cand = [(sr_aic(t, cur + [i], Xm), i) for i in pool_cix if i not in cur]
            if not cand:
                break
            a, i = min(cand)
            if a >= best:
                break
            best, cur = a, cur + [i]
        return cur

    sr_ix6 = [sr_cix[c] for c in EXOG]
    _T = len(sr_dates)

    # Cross-check the fast path against the notebook's own ardl_design(). ardl_design
    # computes y.shift(1) internally, so it must be handed the GAP-FREE calendar series
    # (y_nan, which carries COVID as NaN-in-place with all rows retained) rather than the
    # already-filtered common sample — on a filtered frame that shift would be positional
    # across dropped rows instead of calendar-correct. Both sides are then restricted to
    # an identical row set before comparison.
    _idx_nb, _Y_nb, _M_nb = ardl_design(y_nan, df.loc[y_nan.index, EXOG], 1)
    _keep = _idx_nb.isin(sr_dates)
    _b_nb, *_ = np.linalg.lstsq(_M_nb[_keep], _Y_nb[_keep], rcond=None)
    _pos = [sr_dates.get_loc(d) for d in _idx_nb[_keep]]
    _M_fast = np.column_stack([np.ones(len(_pos)), sr_ylag[_pos],
                               sr_X[np.ix_(_pos, sr_ix6)]])
    _b_fast, *_ = np.linalg.lstsq(_M_fast, sr_y[_pos], rcond=None)
    _dmax = float(np.max(np.abs(_b_fast - _b_nb)))
    print(f'\nFast-path cross-check vs the notebook ardl_design() on {len(_pos)} '
          f'identical rows:')
    print(f'  max |coefficient difference| = {_dmax:.3e}   '
          f'{"PASS" if _dmax < 1e-8 else "FAIL"}')

    # ── Baselines ─────────────────────────────────────────────────────────────
    SR_FIXED = sr_rmse(sr_backtest(sr_ix6))
    SR_RW    = sr_rmse(sr_y[SR_ORIGINS] - sr_ylag[SR_ORIGINS])
    print(f'\n{"Naive random walk":<50s} RMSE = {SR_RW:.4f}')
    print(f'{"Pre-specified block (current design)":<50s} RMSE = {SR_FIXED:.4f}   '
          f'skill vs RW = {100 * (1 - SR_FIXED / SR_RW):.1f}%')

    # ── Regime [1]: exhaustive, scored on realised out-of-sample RMSE ──────────
    SR_ALL63 = sorted((sr_rmse(sr_backtest([sr_cix[c] for c in cs])), cs)
                      for k in range(1, len(EXOG) + 1)
                      for cs in combinations(EXOG, k))
    SR_OOS_BEST = SR_ALL63[0]
    SR_GAIN_REAL = 100 * (1 - SR_OOS_BEST[0] / SR_FIXED)
    sr_rank = 1 + sum(1 for r, _ in SR_ALL63 if r < SR_FIXED)
    print(f'\n[1] CONTAMINATED — subset chosen by realised out-of-sample RMSE '
          f'({len(SR_ALL63)} subsets)')
    print(f'      best RMSE = {SR_OOS_BEST[0]:.4f}   search gain = {SR_GAIN_REAL:+.1f}%   '
          f'k={len(SR_OOS_BEST[1])}')
    print(f'      subset    = {[c.replace("us_", "") for c in SR_OOS_BEST[1]]}')
    print(f'      pre-specified block ranks {sr_rank}/{len(SR_ALL63)}')

    # ── Regime [2]: chosen once by in-sample AIC on the full sample ────────────
    _bb = min((sr_aic(_T, [sr_cix[c] for c in cs], sr_X), cs)
              for k in range(1, len(EXOG) + 1) for cs in combinations(EXOG, k))
    SR_AIC_FULL = sr_rmse(sr_backtest([sr_cix[c] for c in _bb[1]]))
    _sw_full = sr_stepwise(_T, list(range(len(SR_POOL))), sr_X)
    SR_SW_FULL = sr_rmse(sr_backtest(_sw_full))
    print(f'\n[2] CONTAMINATED — subset chosen once by in-sample AIC on the full sample')
    print(f'      exhaustive, pre-specified block : RMSE = {SR_AIC_FULL:.4f}  '
          f'({100 * (1 - SR_AIC_FULL / SR_FIXED):+.1f}%)  k={len(_bb[1])}')
    print(f'      forward stepwise, full pool     : RMSE = {SR_SW_FULL:.4f}  '
          f'({100 * (1 - SR_SW_FULL / SR_FIXED):+.1f}%)')
    print(f'        picked {[SR_POOL[i].replace("us_", "") for i in _sw_full]}')

    # ── Regime [3]: HONEST — re-selected inside every training window ──────────
    def sr_nested(pool_cix, mode='exhaustive'):
        subsets = ([list(cs) for k in range(1, len(pool_cix) + 1)
                    for cs in combinations(pool_cix, k)] if mode == 'exhaustive' else None)
        e, chosen = np.empty(len(SR_ORIGINS)), []
        for j, t in enumerate(SR_ORIGINS):
            if mode == 'exhaustive':
                cix = min(((sr_aic(t, c, sr_X), c) for c in subsets), key=lambda z: z[0])[1]
            else:
                cix = sr_stepwise(t, pool_cix, sr_X)
            M = sr_design(t, cix, sr_X)
            b, *_ = np.linalg.lstsq(M, sr_y[:t], rcond=None)
            e[j] = sr_y[t] - np.concatenate([[1.0, sr_ylag[t]], sr_X[t, cix]]) @ b
            chosen.append(tuple(sorted(cix)))
        return e, chosen

    SR_E_NEST, SR_CH_NEST = sr_nested(sr_ix6, 'exhaustive')
    SR_NEST = sr_rmse(SR_E_NEST)
    SR_E_NESTSW, SR_CH_NESTSW = sr_nested(list(range(len(SR_POOL))), 'stepwise')
    SR_NESTSW = sr_rmse(SR_E_NESTSW)
    print(f'\n[3] HONEST — subset re-chosen by AIC inside every training window')
    print(f'      exhaustive, pre-specified block : RMSE = {SR_NEST:.4f}  '
          f'({100 * (1 - SR_NEST / SR_FIXED):+.1f}% vs fixed)  '
          f'{len(Counter(SR_CH_NEST))} distinct subsets used')
    print(f'      forward stepwise, full pool     : RMSE = {SR_NESTSW:.4f}  '
          f'({100 * (1 - SR_NESTSW / SR_FIXED):+.1f}% vs fixed)  '
          f'{len(Counter(SR_CH_NESTSW))} distinct subsets used')

    # ── Placebo [A]: REPLACEMENT — equal search space, zero true signal ────────
    print(f'\n[A] PLACEBO (primary) — all {len(EXOG)} regressors REPLACED by pure '
          f'standard-normal noise')
    print(f'      identical {len(SR_ALL63)}-subset search; equal search space; zero true '
          f'signal by construction')
    print(f'      -> the gain measured here is pure search artefact')
    print(f'      {"seed":>6s} {"all-noise":>10s} {"best-of-" + str(len(SR_ALL63)):>12s} '
          f'{"search gain":>12s}')
    SR_PLA = []
    for _seed in range(1, SR_SEEDS_A + 1):
        _rg = np.random.default_rng(_seed)
        _Xn = _rg.standard_normal((len(sr_work), len(EXOG)))
        _fn = sr_rmse(sr_backtest(list(range(len(EXOG))), _Xn))
        _bn = min(sr_rmse(sr_backtest(list(cs), _Xn))
                  for k in range(1, len(EXOG) + 1)
                  for cs in combinations(range(len(EXOG)), k))
        SR_PLA.append({'seed': _seed, 'all_noise': _fn, 'best': _bn,
                       'gain': 100 * (1 - _bn / _fn)})
        print(f'      {_seed:6d} {_fn:10.4f} {_bn:12.4f} {100 * (1 - _bn / _fn):+11.1f}%')
    SR_PLA = pd.DataFrame(SR_PLA)
    SR_ART_MEAN, SR_ART_MAX = SR_PLA['gain'].mean(), SR_PLA['gain'].max()
    print(f'\n      artefact gain from searching noise : mean {SR_ART_MEAN:+.1f}%  '
          f'range {SR_PLA["gain"].min():+.1f}%..{SR_ART_MAX:+.1f}%')
    print(f'      real gain from searching regressors: {SR_GAIN_REAL:+.1f}%')
    print(f'      -> roughly {100 * SR_ART_MEAN / SR_GAIN_REAL:.0f}% of the apparent gain '
          f'(up to {100 * SR_ART_MAX / SR_GAIN_REAL:.0f}% on the worst seed) is pure artefact')
    if SR_GAIN_REAL > SR_ART_MAX:
        print(f'      -> the real gain EXCEEDS the artefact band, so genuine predictive')
        print(f'         heterogeneity between subsets does exist. Regime [3] tests whether')
        print(f'         it is capturable without look-ahead.')
    else:
        print(f'      -> the real gain sits INSIDE the artefact band: no evidence of genuine')
        print(f'         heterogeneity between subsets.')

    # ── Placebo [B]: CONTAMINATION — can the search tell signal from noise? ────
    print(f'\n[B] PLACEBO (secondary) — {SR_N_NOISE} noise regressors ADDED alongside the '
          f'real ones')
    print(f'      RMSE here is NOT evidence (superset search can only improve); only the')
    print(f'      selection-frequency statistic below is informative')
    print(f'      {"seed":>6s} {"noise in winner":>16s} {"top-10 subsets w/ noise":>25s}')
    SR_PLB = []
    _nb = len(SR_POOL)
    for _seed in range(1, SR_SEEDS_B + 1):
        _rg = np.random.default_rng(1000 + _seed)
        _Xa = np.column_stack([sr_X, _rg.standard_normal((len(sr_work), SR_N_NOISE))])
        _pool = sr_ix6 + list(range(_nb, _nb + SR_N_NOISE))
        _rs = sorted((sr_rmse(sr_backtest(list(cs), _Xa)), cs)
                     for k in range(1, len(_pool) + 1) for cs in combinations(_pool, k))
        _nz = lambda cs: sum(1 for i in cs if i >= _nb)
        _t10 = sum(1 for _, cs in _rs[:10] if _nz(cs) > 0)
        SR_PLB.append({'seed': _seed, 'n_noise_winner': _nz(_rs[0][1]), 'top10_noise': _t10})
        print(f'      {_seed:6d} {_nz(_rs[0][1]):16d} {str(_t10) + "/10":>25s}')
    SR_PLB = pd.DataFrame(SR_PLB)
    print(f'\n      mean noise regressors in winning subset : '
          f'{SR_PLB["n_noise_winner"].mean():.1f} of {SR_N_NOISE}')
    print(f'      mean top-10 subsets containing noise    : '
          f'{SR_PLB["top10_noise"].mean():.1f}/10')
    print(f'      -> a search scored on realised RMSE cannot reliably distinguish an')
    print(f'         informative regressor from a random one at this sample size')

    # ── Summary ───────────────────────────────────────────────────────────────
    SR_SUMMARY = pd.DataFrame([
        ('Naive random walk',                    '—',                          'n/a',    SR_RW),
        ('Pre-specified block (current design)', 'fixed a priori',             'none',   SR_FIXED),
        ('Exhaustive, scored on OOS RMSE',       'realised OOS RMSE',          'severe', SR_OOS_BEST[0]),
        ('Exhaustive, full-sample AIC',          'in-sample AIC (once)',       'yes',    SR_AIC_FULL),
        ('Forward stepwise, full-sample AIC',    'in-sample AIC (once)',       'yes',    SR_SW_FULL),
        ('Exhaustive, nested per origin',        'in-sample AIC (per origin)', 'none',   SR_NEST),
        ('Forward stepwise, nested per origin',  'in-sample AIC (per origin)', 'none',   SR_NESTSW),
    ], columns=['Approach', 'Selection basis', 'Look-ahead', 'RMSE'])
    SR_SUMMARY['vs fixed (%)'] = (100 * (1 - SR_SUMMARY['RMSE'] / SR_FIXED)).round(1)
    SR_SUMMARY['RMSE'] = SR_SUMMARY['RMSE'].round(4)
    print('\n' + '=' * 94)
    print('SPECIFICATION-SEARCH ROBUSTNESS SUMMARY   '
          '(positive % = lower RMSE than the pre-specified block)')
    print('=' * 94)
    print(SR_SUMMARY.to_string(index=False))
    print('-' * 94)
    print(f'Search artefact benchmark (placebo A): a {len(SR_ALL63)}-subset search over pure '
          f'noise\n  manufactures {SR_ART_MEAN:+.1f}% mean / {SR_ART_MAX:+.1f}% worst-case '
          f'"gain" with zero true signal.')
    print(f'CONCLUSION: the only regimes without look-ahead are the last two, and both sit '
          f'at\n  or behind the pre-specified block. Searching yields nothing achievable in '
          f'production.')
    print('=' * 94)

    # ── LaTeX export (in-notebook only unless SAVE_LATEX_TABLE) ───────────────
    def sr_to_latex(d):
        esc = lambda s: str(s).replace('%', r'\%').replace('_', r'\_').replace('—', '--')
        L = [r'\begin{table}[h]', r'\centering',
             r'\caption{Specification-search robustness: the pre-specified exogenous block '
             r'versus searched alternatives}',
             r'\label{tab:spec-search-robustness}',
             r'\begin{tabular}{@{}lllcc@{}}', r'\toprule',
             r'\textbf{Approach} & \textbf{Selection basis} & \textbf{Look-ahead} & '
             r'RMSE & vs fixed (\%) \\', r'\midrule']
        for _, r in d.iterrows():
            L.append(f'{esc(r["Approach"])} & {esc(r["Selection basis"])} & '
                     f'{esc(r["Look-ahead"])} & {r["RMSE"]:.4f} & {r["vs fixed (%)"]:+.1f} \\\\')
        L += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
        return '\n'.join(L)

    sr_latex = sr_to_latex(SR_SUMMARY)
    if SAVE_LATEX_TABLE:
        try:
            (LOCAL_OUTPUT / f'{MODEL_PREFIX}_spec_search_robustness.tex').write_text(sr_latex)
            print(f'\nLaTeX table -> '
                  f'{LOCAL_OUTPUT / (MODEL_PREFIX + "_spec_search_robustness.tex")}')
        except Exception as e:
            print(f'\n(LaTeX not written to disk: {e})')
    else:
        print('\nLaTeX table (disk export OFF — set SAVE_LATEX_TABLE=True to re-enable):')
    print(sr_latex)
else:
    print('Section 3b skipped (RUN_SEARCH_ROBUSTNESS = False).')

Candidate pool: 13 lagged regressors (6 pre-specified, 7 additional)
    + us_cpi_L0
    + us_bond_yield_10y_d1_L2
    + us_sp500_log_ret_L4
    + us_indprod_yoy_L3
    + us_reer_diff_L0
    + us_oil_yoy_L2
    + us_vix_log_ret_L0

Common estimation sample : 1994-06-30 -> 2025-12-31  (n=116)
  Section 3 headline was : 1991-12-31 -> 2025-12-31  (n=137)   <- RMSEs NOT cross-comparable
  origins                : 55  (2009-06-30 -> 2025-12-31)

Fast-path cross-check vs the notebook ardl_design() on 116 identical rows:
  max |coefficient difference| = 0.000e+00   PASS

Naive random walk                                  RMSE = 0.1898
Pre-specified block (current design)               RMSE = 0.1546   skill vs RW = 18.6%

[1] CONTAMINATED — subset chosen by realised out-of-sample RMSE (63 subsets)
      best RMSE = 0.1097   search gain = +29.0%   k=2
      subset    = ['consumer_confidence_L2', 'credit_qoq_growth_L6']
      pre-specified block ranks 25/63

[2] CONTAMINATED — subset chosen once

In [8]:
# ── Figures 3b.1-3b.5 — visual validation of the specification-search finding ──
if RUN_SEARCH_ROBUSTNESS:

    _all_r = np.array([r for r, _ in SR_ALL63])
    _od = [sr_dates[t] for t in SR_ORIGINS]

    # ── Figure 3b.1 — where the pre-specified block sits in the search space ───
    fig = make_subplots(
        rows=1, cols=2, column_widths=[0.58, 0.42],
        subplot_titles=(f'Out-of-sample RMSE across all {len(SR_ALL63)} subsets',
                        'RMSE by number of regressors retained'))
    fig.add_trace(go.Histogram(x=_all_r, nbinsx=24, marker_color=BLUE,
                               opacity=0.75, showlegend=False), row=1, col=1)
    for _x, _c, _lbl, _dash in [(SR_FIXED, NAVY, 'Pre-specified block', 'solid'),
                                (SR_OOS_BEST[0], RED, 'Best (chosen on OOS RMSE)', 'dash'),
                                (SR_RW, GREY, 'Naive random walk', 'dot')]:
        fig.add_vline(x=_x, line=dict(color=_c, width=2, dash=_dash), row=1, col=1)
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                                 line=dict(color=_c, width=2, dash=_dash), name=_lbl),
                      row=1, col=1)
    fig.add_trace(go.Box(x=[len(cs) for _, cs in SR_ALL63], y=_all_r, marker_color=BLUE,
                         showlegend=False, boxpoints='all', jitter=0.4, pointpos=0,
                         marker_size=3), row=1, col=2)
    fig.add_hline(y=SR_FIXED, line=dict(color=NAVY, width=2), row=1, col=2)
    fig.update_xaxes(title_text='out-of-sample RMSE', row=1, col=1)
    fig.update_yaxes(title_text='number of subsets', row=1, col=1)
    fig.update_xaxes(title_text='regressors retained (k)', row=1, col=2)
    fig.update_yaxes(title_text='out-of-sample RMSE', row=1, col=2)
    fig.update_layout(
        title=dict(text='Figure 3b.1 — The pre-specified block within the search space<br>'
                        f'<sup>Scored on realised out-of-sample RMSE, the pre-specified block '
                        f'ranks {sr_rank} of {len(SR_ALL63)}. A better-looking subset always '
                        'exists; Figures 3b.2 and 3b.3 test whether it can be found without '
                        'look-ahead.</sup>',
                   font=dict(size=13, color=NAVY)),
        template=TEMPLATE, height=430, bargap=0.05,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1,
                    font=dict(size=10)),
        margin=dict(t=115, b=50, l=65, r=20))
    fig.update_annotations(font_size=10)
    fig.show()

    # ── Figure 3b.2 — replacement placebo: how much gain is pure artefact ─────
    fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45],
                        subplot_titles=('Gain from searching: real regressors vs pure noise',
                                        'Can the search tell signal from noise?'),
                        horizontal_spacing=0.13)
    fig.add_trace(go.Box(y=SR_PLA['gain'], name='Pure noise<br>(placebo A)',
                         marker_color=AMBER, boxpoints='all', jitter=0.5, pointpos=0,
                         marker_size=6, showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=['Pure noise<br>(placebo A)'], y=[SR_GAIN_REAL],
                             mode='markers+text', marker=dict(color=RED, size=15,
                             symbol='diamond', line=dict(color='white', width=1.5)),
                             text=[f'  real regressors: {SR_GAIN_REAL:+.1f}%'],
                             textposition='middle right',
                             textfont=dict(size=10, color=RED), showlegend=False),
                  row=1, col=1)
    fig.add_hline(y=0, line=dict(color=NAVY, width=1.5), row=1, col=1)
    fig.add_hline(y=SR_ART_MAX, line=dict(color=AMBER, width=1.5, dash='dot'), row=1, col=1)
    fig.add_trace(go.Bar(x=SR_PLB['seed'].astype(str), y=SR_PLB['top10_noise'],
                         marker_color=AMBER, name='top-10 with noise',
                         text=[f'{v}/10' for v in SR_PLB['top10_noise']],
                         textposition='outside', textfont=dict(size=9), showlegend=False),
                  row=1, col=2)
    fig.add_trace(go.Scatter(x=SR_PLB['seed'].astype(str), y=SR_PLB['n_noise_winner'],
                             mode='markers+lines', marker=dict(color=RED, size=9),
                             line=dict(color=RED, width=1.5, dash='dash'),
                             name='noise in winning subset'), row=1, col=2)
    fig.update_yaxes(title_text='RMSE gain from searching (%)', row=1, col=1)
    fig.update_xaxes(title_text='placebo seed', row=1, col=2)
    fig.update_yaxes(title_text='count', range=[0, 11], row=1, col=2)
    fig.update_layout(
        title=dict(text='Figure 3b.2 — Placebo tests: separating real signal from search '
                        'artefact<br>'
                        f'<sup>Left (placebo A, primary): all {len(EXOG)} regressors replaced '
                        f'by pure standard-normal noise, identical {len(SR_ALL63)}-subset '
                        f'search over {SR_SEEDS_A} seeds. An equally sized search over noise '
                        f'manufactures {SR_ART_MEAN:+.1f}% mean gain from nothing, so roughly '
                        f'{100 * SR_ART_MEAN / SR_GAIN_REAL:.0f}% of the real '
                        f'{SR_GAIN_REAL:+.1f}% is artefact. Right (placebo B, secondary): with '
                        f'{SR_N_NOISE} noise regressors added, noise still enters the '
                        'top-ranked subsets — the search cannot reliably tell them '
                        'apart.</sup>',
                   font=dict(size=13, color=NAVY)),
        template=TEMPLATE, height=470,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1,
                    font=dict(size=9)),
        margin=dict(t=150, b=50, l=70, r=30))
    fig.update_annotations(font_size=10)
    fig.show()

    # ── Figure 3b.3 — the look-ahead ladder ───────────────────────────────────
    _lad = SR_SUMMARY[SR_SUMMARY['Approach'] != 'Naive random walk'].copy()
    _col = {'none': GREEN, 'yes': AMBER, 'severe': RED, 'n/a': GREY}
    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=_lad['Approach'], x=_lad['RMSE'], orientation='h',
        marker_color=[_col[l] for l in _lad['Look-ahead']],
        text=[f'{r:.4f}   ({g:+.1f}%)' for r, g in zip(_lad['RMSE'], _lad['vs fixed (%)'])],
        textposition='outside', textfont=dict(size=9), showlegend=False))
    fig.add_vline(x=SR_FIXED, line=dict(color=NAVY, width=2, dash='dash'))
    fig.add_vline(x=SR_RW, line=dict(color=GREY, width=1.5, dash='dot'),
                  annotation_text='naive random walk', annotation_position='top',
                  annotation_font=dict(size=9, color=GREY))
    for _lbl, _c in [('no look-ahead (achievable in production)', GREEN),
                     ('selected once on the full sample', AMBER),
                     ('selected on the reported metric', RED)]:
        fig.add_trace(go.Bar(y=[None], x=[None], orientation='h',
                             marker_color=_c, name=_lbl))
    fig.update_layout(
        title=dict(text='Figure 3b.3 — Apparent accuracy collapses once look-ahead is '
                        'removed<br>'
                        '<sup>Dashed navy line = pre-specified block. The two green bars are '
                        'the only regimes reproducible in production, and both sit at or '
                        'behind it: the heterogeneity the search finds is real but not '
                        'capturable ex ante.</sup>',
                   font=dict(size=13, color=NAVY)),
        template=TEMPLATE, height=450,
        xaxis_title='out-of-sample RMSE',
        xaxis_range=[0, max(_lad['RMSE']) * 1.38],
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1,
                    font=dict(size=9)),
        margin=dict(t=118, b=50, l=280, r=30))
    fig.show()

    # ── Figure 3b.4 — what honest selection actually keeps ────────────────────
    _M = np.zeros((len(EXOG), len(SR_CH_NEST)))
    for j, cs in enumerate(SR_CH_NEST):
        for i in cs:
            _M[sr_ix6.index(i), j] = 1
    _freq = _M.mean(axis=1)
    fig = make_subplots(rows=1, cols=2, column_widths=[0.72, 0.28],
                        subplot_titles=('Regressor retained at each forecast origin',
                                        'Retention frequency'),
                        horizontal_spacing=0.10)
    fig.add_trace(go.Heatmap(
        z=_M, x=_od, y=[c.replace('us_', '') for c in EXOG],
        colorscale=[[0, '#EEF2F7'], [1, NAVY]], showscale=False,
        hovertemplate='%{y}<br>%{x|%Y-%m}<br>retained=%{z}<extra></extra>'), row=1, col=1)
    fig.add_trace(go.Bar(y=[c.replace('us_', '') for c in EXOG], x=_freq * 100,
                         orientation='h', marker_color=BLUE, showlegend=False,
                         text=[f'{f * 100:.0f}%' for f in _freq],
                         textposition='outside', textfont=dict(size=9)), row=1, col=2)
    fig.update_xaxes(title_text='forecast origin', row=1, col=1)
    fig.update_xaxes(title_text='% of origins retained', range=[0, 120], row=1, col=2)
    fig.update_layout(
        title=dict(text='Figure 3b.4 — Which regressors honest selection keeps<br>'
                        f'<sup>Exhaustive Akaike-information-criterion selection re-run inside '
                        f'every training window: {len(Counter(SR_CH_NEST))} distinct subsets '
                        f'across {len(SR_CH_NEST)} origins. Regressors retained at nearly every '
                        'origin are robustly supported; intermittent ones are the selection '
                        'criterion chasing sample noise.</sup>',
                   font=dict(size=13, color=NAVY)),
        template=TEMPLATE, height=410,
        margin=dict(t=120, b=50, l=180, r=60))
    fig.update_annotations(font_size=10)
    fig.show()

    # ── Figure 3b.5 — cumulative squared error through time ──────────────────
    _e_fixed = sr_backtest(sr_ix6)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.62, 0.38],
                        vertical_spacing=0.08,
                        subplot_titles=('Cumulative squared forecast error (lower is better)',
                                        'Cumulative advantage of the pre-specified block'))
    for _e, _lbl, _c, _d in [
            (sr_y[SR_ORIGINS] - sr_ylag[SR_ORIGINS], 'Naive random walk', GREY, 'dot'),
            (_e_fixed, 'Pre-specified block', NAVY, 'solid'),
            (SR_E_NEST, 'Nested selection (exhaustive AIC)', GREEN, 'dash'),
            (SR_E_NESTSW, 'Nested selection (forward stepwise)', AMBER, 'dashdot')]:
        fig.add_trace(go.Scatter(x=_od, y=np.cumsum(_e ** 2), mode='lines',
                                 line=dict(color=_c, width=1.8, dash=_d), name=_lbl),
                      row=1, col=1)
    _diff = np.cumsum(SR_E_NEST ** 2) - np.cumsum(_e_fixed ** 2)
    fig.add_trace(go.Scatter(x=_od, y=_diff, mode='lines', fill='tozeroy',
                             line=dict(color=GREEN, width=1.5),
                             fillcolor='rgba(39,174,96,0.18)', showlegend=False),
                  row=2, col=1)
    fig.add_hline(y=0, line=dict(color=GREY, width=1), row=2, col=1)
    for _s, _e_, _l in RECESSIONS_US:
        if pd.Timestamp(_e_) >= _od[0]:
            for _r in (1, 2):
                fig.add_vrect(x0=_s, x1=_e_, fillcolor='lightgrey', opacity=0.3,
                              layer='below', line_width=0, row=_r, col=1)
    fig.update_yaxes(title_text='cumulative squared error', row=1, col=1)
    fig.update_yaxes(title_text='nested − pre-specified', row=2, col=1)
    fig.update_xaxes(title_text='forecast origin', row=2, col=1)
    fig.update_layout(
        title=dict(text='Figure 3b.5 — No sustained period in which searching beats the '
                        'pre-specified block<br>'
                        '<sup>In the lower panel, values above zero mean the pre-specified '
                        'block is ahead on cumulative squared error. Grey bands = recessions; '
                        'coronavirus-window targets are excluded from scoring.</sup>',
                   font=dict(size=13, color=NAVY)),
        template=TEMPLATE, height=590,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1,
                    font=dict(size=9)),
        margin=dict(t=118, b=50, l=75, r=30))
    fig.update_annotations(font_size=10)
    fig.show()

### Specification-Search Findings

Figures below are from the run reported in the cell output above; they recompute on
execution and the printed output is authoritative. The common-sample requirement gives
n = 116 and 55 origins here, shorter than Section 3's 67 origins, so these RMSEs are
**not** comparable to the Section 3 table.

**Searching does find something real — but it cannot be acted upon.** Scored on realised
out-of-sample RMSE, the best of the 63 subsets improves on the pre-specified block by
29.0% (0.1097 versus 0.1546), and the pre-specified block ranks only 25th of 63. Taken
alone that looks like a strong case for searching. Placebo A shows that an identically
sized search over *pure noise* manufactures +5.7% mean (+10.7% worst-case) improvement
from nothing, so roughly a fifth of the apparent gain — up to 37% of it on the worst seed
— is pure artefact. The real gain does exceed that artefact band, which means genuine
predictive heterogeneity between subsets exists; this is **not** a case of the search
finding nothing at all.

**The decisive result is regime 3.** When the subset must be re-chosen at each forecast
origin using only data available at that point, the realised gain does not merely shrink,
it reverses: exhaustive AIC selection returns 0.1579, some 2.1% *worse* than simply fixing
the block, and forward stepwise 0.1604, 3.8% worse. The heterogeneity is real, but the
identity of the better-performing subset is not learnable from the training sample at any
origin, so none of it is capturable ex ante. Figure 3b.5 confirms there is no sustained
sub-period in which searching pulls ahead; Figure 3b.4 shows why, with honest selection
churning through four distinct subsets rather than converging on one.

**Stepwise selection specifically.** Forward stepwise over the full 13-regressor pool
looks like a +5.5% improvement when fitted once on the whole sample, against +8.5% for
the exhaustive search — so the greedy path gives up accuracy without escaping the
selection bias. Under honest nested selection it is the worst of the four search variants
tested at 3.8% behind the fixed block. Stepwise here therefore carries the contamination
of a full search while delivering accuracy below a fixed block. Placebo B reinforces the
point: noise regressors appear in 7.0 of the top 10 subsets on average and in 0.8 of the
winning subsets, so a search scored at this sample size cannot reliably tell an
informative regressor from a random one (Freedman 1983; Harrell 2015).

**Conclusion for the pipeline.** The pre-specified CCF block is retained as the headline
specification. The justification is not that alternatives are unexplored — they are
exhaustively explored here — but that no alternative delivers a reproducible improvement
once look-ahead is removed, and that keeping the block identical to the Stage 2 regressor
set preserves the like-for-like comparison this notebook exists to provide.

**Two acknowledgements, recorded rather than buried.** First, the Diebold-Mariano and
Clark-West statistics in Section 3 presuppose a single specification fixed in advance;
they are *not* valid as multiple-specification inference, and the numbers in this section
must not be read as significance tests. Formal inference across many specifications
requires White's (2000) Reality Check, Hansen's (2005) test for superior predictive
ability, or the Model Confidence Set of Hansen, Lunde and Nason (2011). Second, the same
critique applies in kind — though at much smaller scale — to the `auto_arima` order
selection in Section 2, which is chosen once on the full sample. That is the
specification look-ahead bias already flagged in the report; the candidate space there is
a small nested lattice of low-order specifications rather than a combinatorial subset
space, and Clark-West specifically addresses the random-walk nesting, but the mechanism
is the same and is disclosed on that basis (Leeb & Pötscher 2005).

## **4: 20-Quarter Forward Forecast (2026–2030)**
___
Each model is re-fit on the full NaN-in-place history and projected 20 quarters. The
conditional models (SARIMAX, ARDL) are driven by **02a's forecasted regressors**; the
VAR generates its own paths. A 95% band is shown for the SARIMAX conditional mean only
— and, per the note in Cell 0, that band still treats the upstream macro forecasts as
known, so it is a *lower bound* on true forecast uncertainty. Full uncertainty
propagation and IFRS 9 scenario construction belong to a later stage.

Note the regressors phase in as their lags reach the horizon: only `us_gdp_yoy_growth_L0`
is forecast-dependent from 2026Q1; unemployment (L1) from 2026Q2, confidence (L2) from
2026Q3, house prices (L3) from 2026Q4, and credit/CPI (L6) from 2027Q3.

In [9]:
# ── SARIMAX/AR/SARIMA forward via state space (NaN-in-place history) ───────────
def sx_forward(order, sord, exog_hist, exog_fc):
    m = SARIMAX(y_nan, exog=exog_hist, order=order, seasonal_order=sord,
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
    fo = m.get_forecast(H, exog=exog_fc); ci = fo.conf_int(alpha=0.05)
    return fo.predicted_mean.values, ci.iloc[:, 0].values, ci.iloc[:, 1].values

fwd = {}; ci_lo = {}; ci_hi = {}
fwd['Naive RW'] = np.repeat(y_full.dropna().iloc[-1], H)
fwd['AR(1)'],   ci_lo['AR(1)'],   ci_hi['AR(1)']   = sx_forward((1, 0, 0), (0, 0, 0, 0), None, None)
fwd['SARIMA'],  ci_lo['SARIMA'],  ci_hi['SARIMA']  = sx_forward(SARIMA_ORD, SARIMA_SORD, None, None)
fwd['SARIMAX'], ci_lo['SARIMAX'], ci_hi['SARIMAX'] = sx_forward(SARIMAX_ORD, SARIMAX_SORD, X_hist, X_fc)

# ARDL forward (recursive, hand-rolled)
b, _ = ardl_fit(y_nan, X_hist, ARDL_P)
fwd['ARDL'] = ardl_forecast(b, y_nan, X_fc, ARDL_P)

# VAR forward (joint system; reconstruct level if differenced)
Vm = V0.diff().dropna() if VAR_DIFF else V0
vres = VAR(Vm).fit(maxlags=4, ic='aic')
vf = pd.DataFrame(vres.forecast(Vm.values[-vres.k_ar:], steps=H), columns=Vm.columns)[TARGET].values
if VAR_DIFF: vf = V0[TARGET].iloc[-1] + np.cumsum(vf)
fwd['VAR'] = vf

forward = pd.DataFrame(fwd, index=fc_idx)
last_obs = float(y_full.dropna().iloc[-1])
print(f'Last observed delinquency ({y_full.dropna().index[-1].date()}): {last_obs:.3f}%\n')
print('20-quarter forward summary (delinquency %):')
print(f'  {"model":<10s} {"2026Q1":>8s} {"2030Q4":>8s} {"min":>7s} {"mean":>7s} {"max":>7s}')
for m in MODELS:
    v = forward[m].values
    print(f'  {m:<10s} {v[0]:8.3f} {v[-1]:8.3f} {v.min():7.3f} {v.mean():7.3f} {v.max():7.3f}')
print(f'\nSpread at 2030Q4: ARDL {forward["ARDL"].iloc[-1]:.2f}%  (highest)  vs  '
      f'VAR {forward["VAR"].iloc[-1]:.2f}%  (lowest)  — a ~{forward["ARDL"].iloc[-1] - forward["VAR"].iloc[-1]:.1f} pp gap.')
print('This ARDL-vs-VAR divergence is the model-choice risk to flag for scenario work: '
      'the exog-driven ARDL extrapolates the recent rise, while the joint VAR mean-reverts.')

Last observed delinquency (2025-12-31): 2.940%

20-quarter forward summary (delinquency %):
  model        2026Q1   2030Q4     min    mean     max
  Naive RW      2.940    2.940   2.940   2.940   2.940
  AR(1)         2.921    2.589   2.589   2.752   2.921
  SARIMA        2.902    2.817   2.774   2.810   2.902
  SARIMAX       2.935    3.016   2.898   2.978   3.016
  ARDL          2.967    3.345   2.798   3.211   3.465
  VAR           2.836    2.309   2.309   2.440   2.836

Spread at 2030Q4: ARDL 3.35%  (highest)  vs  VAR 2.31%  (lowest)  — a ~1.0 pp gap.
This ARDL-vs-VAR divergence is the model-choice risk to flag for scenario work: the exog-driven ARDL extrapolates the recent rise, while the joint VAR mean-reverts.


In [10]:
# Figure 3 — forward forecasts (last 10y history + 20q horizon), SARIMAX band shown
hist_tail = y_full.dropna()
cutoff = hist_tail.index[-1] - pd.DateOffset(years=10)
hist_tail = hist_tail[hist_tail.index >= cutoff]

fig = go.Figure()
fig.add_vrect(x0=COVID_START, x1=COVID_END, fillcolor=AMBER, opacity=0.10, layer='below', line_width=0)
fig.add_trace(go.Scatter(x=hist_tail.index, y=hist_tail.values, mode='lines',
                         line=dict(color=NAVY, width=2.0), name='Historical'))

# SARIMAX 95% band (conditional mean only; see caveat)
fig.add_trace(go.Scatter(x=list(fc_idx) + list(fc_idx[::-1]),
                         y=list(ci_hi['SARIMAX']) + list(ci_lo['SARIMAX'][::-1]),
                         fill='toself', fillcolor='rgba(46,117,182,0.15)',
                         line=dict(width=0), hoverinfo='skip', showlegend=True,
                         name='SARIMAX 95% (macro known)'))
for m in MODELS:
    fig.add_trace(go.Scatter(x=fc_idx, y=forward[m].values, mode='lines',
                             line=dict(color=colmap[m], width=1.8,
                                       dash='dot' if m == 'Naive RW' else 'solid'),
                             name=m))
fig.update_layout(
    title=dict(text='Figure 3 — Delinquency: 20-quarter forward forecast (2026–2030)<br>'
                    '<sup>Last 10y history | shaded band = SARIMAX 95% treating 02a macro '
                    'forecasts as known (lower bound on true uncertainty)</sup>',
               font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=480, yaxis_title='Delinquency rate (%)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1, font=dict(size=10)),
    margin=dict(t=100, b=40, l=55, r=20))
fig.show()

In [11]:
# ── Forecast Range Summary table (Min/Mean/Max + averaged 95% CI) ─────────────
# CI is only available for the three state-space models that route through
# sx_forward() -> get_forecast().conf_int(): AR(1), SARIMA, SARIMAX. ARDL (hand-
# rolled OLS) and VAR (VAR.forecast) carry no interval in this notebook, and
# Naive RW is a deterministic repeat -> all three show '--' for CI.
summary_rows = []
for m in MODELS:
    v = forward[m].values
    if m in ci_lo:                          # AR(1), SARIMA, SARIMAX only
        ci_lower_mean = ci_lo[m].mean()
        ci_upper_mean = ci_hi[m].mean()
    else:                                    # Naive RW, ARDL, VAR
        ci_lower_mean = np.nan
        ci_upper_mean = np.nan
    summary_rows.append({
        'Model': m,
        'Min (%)': v.min(), 'Mean (%)': v.mean(), 'Max (%)': v.max(),
        'CI lower (%)': ci_lower_mean, 'CI upper (%)': ci_upper_mean,
    })

forecast_range_summary = pd.DataFrame(summary_rows).set_index('Model').round(3)
print('Forecast Range Summary (2026-2030):')
print(forecast_range_summary.to_string())

# ── booktabs LaTeX export (table stays in-notebook only; no disk write) ───────
def forecast_range_to_latex(df, caption, label):
    esc = lambda s: str(s).replace('%', r'\%').replace('_', r'\_')
    n = len(df.columns)
    header = ' & '.join(esc(c) for c in df.columns)
    lines = [
        r'\begin{table}[h]', r'\centering',
        f'\\caption{{{caption}}}', f'\\label{{{label}}}',
        r'\begin{tabular}{@{}l' + 'c' * n + r'@{}}', r'\toprule',
        f'\\textbf{{Model}} & {header} \\\\', r'\midrule',
    ]
    for model, row in df.iterrows():
        vals = ' & '.join('--' if pd.isna(v) else f'{v:.3f}' for v in row)
        lines.append(f'{esc(model)} & {vals} \\\\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)

latex_table = forecast_range_to_latex(
    forecast_range_summary,
    caption='Forecast Range Summary, 2026--2030',
    label='tab:stagea-ts-forecast',
)
print('\n' + latex_table)

Forecast Range Summary (2026-2030):
          Min (%)  Mean (%)  Max (%)  CI lower (%)  CI upper (%)
Model                                                           
Naive RW    2.940     2.940    2.940           NaN           NaN
AR(1)       2.589     2.752    2.921         1.532         3.973
SARIMA      2.774     2.810    2.902         1.101         4.518
SARIMAX     2.898     2.978    3.016         1.609         4.346
ARDL        2.798     3.211    3.465           NaN           NaN
VAR         2.309     2.440    2.836           NaN           NaN

\begin{table}[h]
\centering
\caption{Forecast Range Summary, 2026--2030}
\label{tab:stagea-ts-forecast}
\begin{tabular}{@{}lccccc@{}}
\toprule
\textbf{Model} & Min (\%) & Mean (\%) & Max (\%) & CI lower (\%) & CI upper (\%) \\
\midrule
Naive RW & 2.940 & 2.940 & 2.940 & -- & -- \\
AR(1) & 2.589 & 2.752 & 2.921 & 1.532 & 3.973 \\
SARIMA & 2.774 & 2.810 & 2.902 & 1.101 & 4.518 \\
SARIMAX & 2.898 & 2.978 & 3.016 & 1.609 & 4.346 \\
ARDL & 2.79

### 20-Quarter Forward Forecast Summary (2026 Q1 – 2030 Q4)

Re-fit on the full NaN-in-place history, from a last observed delinquency rate of
**2.940%** (2025 Q4):

| Model | 2026 Q1 (%) | 2030 Q4 (%) | Min (%) | Mean (%) | Max (%) |
|---|---|---|---|---|---|
| Naive RW | 2.940 | 2.940 | 2.940 | 2.940 | 2.940 |
| AR(1) | 2.921 | 2.589 | 2.589 | 2.752 | 2.921 |
| SARIMA | 2.903 | 2.820 | 2.770 | 2.809 | 2.903 |
| SARIMAX | 2.935 | 3.016 | 2.898 | 2.978 | 3.016 |
| ARDL | 2.967 | 3.345 | 2.798 | 3.211 | 3.465 |
| VAR | 2.836 | 2.309 | 2.309 | 2.440 | 2.836 |

The six paths diverge substantially over the five-year lifetime horizon despite starting
within 0.13pp of each other at 2026 Q1. By 2030 Q4, ARDL is highest at 3.345% and VAR
lowest at 2.309% — a **~1.0pp spread** driven almost entirely by how each model treats
the exogenous macro path: ARDL is exogenous-driven and extrapolates the post-2022 rise
embedded in the Stage 1 forecasted regressors, while the joint-system VAR generates its
own macro paths and mean-reverts the level downward instead. AR(1) and SARIMA, carrying
no exogenous information, settle toward their own autoregressive dynamics rather than
tracking either macro narrative. SARIMAX sits between them, drifting mildly upward to
3.016%.

This ARDL-versus-VAR spread is the single largest source of model-choice risk for the
downstream scenario-construction stage: whichever forward path — or blend — is eventually
used as the IFRS 9 base-case macro-conditional default-rate path will materially shape
the projected expected credit loss, and that choice is **not** resolved by the backtest
above. Section 3 showed ARDL and SARIMA to be statistically indistinguishable at h = 1,
so nothing in this notebook's evidence base adjudicates between those two five-year
paths. Note also that SARIMAX, which posts near-zero realised skill in the backtest,
nonetheless produces one of the more benign forward paths — a reminder that short-horizon
accuracy and long-horizon path plausibility are separate properties.

## **5: Export & Limitations**
___
The forward paths are written to `TS_delinquency_forecast_US_Q.csv` for comparison
against the ML PD model. This is the time-series **benchmark** deliverable — not a
replacement for the ML model, and not the final ECL scenario set.

In [12]:
# ── Save forward forecasts + SARIMAX band ─────────────────────────────────────
out = forward.copy()
out['SARIMAX_lo95'] = ci_lo['SARIMAX']
out['SARIMAX_hi95'] = ci_hi['SARIMAX']
out.index.name = 'date'

if SAVE_CSV_OUTPUTS:
    saved_to = None
    for target_dir in [LOCAL_OUTPUT, Path.cwd()]:
        try:
            target_dir.mkdir(parents=True, exist_ok=True)
            out.to_csv(target_dir / OUT_FORECAST); saved_to = target_dir / OUT_FORECAST; break
        except Exception:
            continue
    print(f'Saved {OUT_FORECAST}  {out.shape}  ->  {saved_to}')
else:
    print(f'CSV export OFF (set SAVE_CSV_OUTPUTS=True to re-enable)  |  {OUT_FORECAST}  {out.shape}')
print(out.round(3).to_string())

print('''
─────────────────────────────────────────────────────────────────────────────
LIMITATIONS (state in the write-up)
─────────────────────────────────────────────────────────────────────────────
1. Exogenous uncertainty not propagated. SARIMAX/ARDL treat 02a's forecasted
   regressors as known; the plotted band is therefore a LOWER BOUND on true
   forecast uncertainty. Propagation + IFRS 9 base/upside/downside are a later stage.
2. One-step backtest != five-year path. h=1 uses realised lagged regressors, so it
   validates short-horizon conditional skill, not the full lifetime projection.
3. COVID excluded (fit + eval). The 2020–21 forbearance break inverted the macro→
   credit link; those quarters are NaN and skipped, not modelled.
4. VAR is an independent cross-check, not a consumer of Stage 1. Its divergence from
   ARDL bounds model-choice risk for the scenario stage.
5. Specification chosen once. Orders are selected on the full dropped sample and only
   re-estimated per origin — mildly optimistic RMSE (re-run auto_arima in-loop to remove).
6. US proof-of-concept. The UK pipeline requires re-sourcing the UK target and re-running.
''')

CSV export OFF (set SAVE_CSV_OUTPUTS=True to re-enable)  |  TS_delinquency_forecast_US_Q.csv  (20, 8)
            Naive RW  AR(1)  SARIMA  SARIMAX   ARDL    VAR  SARIMAX_lo95  SARIMAX_hi95
date                                                                                  
2026-03-31      2.94  2.921   2.902    2.935  2.967  2.836         2.630         3.240
2026-06-30      2.94  2.903   2.849    2.898  2.801  2.702         2.406         3.390
2026-09-30      2.94  2.885   2.825    2.905  2.798  2.597         2.242         3.568
2026-12-31      2.94  2.866   2.823    2.942  2.851  2.551         2.110         3.773
2027-03-31      2.94  2.848   2.801    2.956  2.955  2.509         2.005         3.908
2027-06-30      2.94  2.830   2.774    2.953  3.051  2.466         1.887         4.020
2027-09-30      2.94  2.812   2.776    2.973  3.091  2.433         1.794         4.151
2027-12-31      2.94  2.794   2.785    2.994  3.138  2.419         1.707         4.280
2028-03-31      2.94  2.777 

### References

Box, G.E.P. & Jenkins, G.M. (1976). *Time Series Analysis: Forecasting and
Control.* Holden-Day.

Clark, T.E. & West, K.D. (2007). Approximately normal tests for equal
predictive accuracy in nested models. *Journal of Econometrics*, 138(1),
291–311.

Dickey, D.A. & Fuller, W.A. (1979). Distribution of the estimators for
autoregressive time series with a unit root. *Journal of the American
Statistical Association*, 74(366a), 427–431.

Diebold, F.X. & Mariano, R.S. (1995). Comparing predictive accuracy.
*Journal of Business & Economic Statistics*, 13(3), 253–263.

Freedman, D.A. (1983). A note on screening regression equations.
*The American Statistician*, 37(2), 152–155.

Hamilton, J.D. (1994). *Time Series Analysis.* Princeton University Press.

Hansen, P.R. (2005). A test for superior predictive ability. *Journal of
Business & Economic Statistics*, 23(4), 365–380.

Hansen, P.R., Lunde, A. & Nason, J.M. (2011). The model confidence set.
*Econometrica*, 79(2), 453–497.

Harrell, F.E. (2015). *Regression Modeling Strategies*, 2nd edn. Springer.

Harvey, D., Leybourne, S. & Newbold, P. (1997). Testing the equality of
prediction mean squared errors. *International Journal of Forecasting*,
13(2), 281–291.

Hyndman, R.J. & Khandakar, Y. (2008). Automatic time series forecasting:
the forecast package for R. *Journal of Statistical Software*, 27(3), 1–22.

Leeb, H. & Pötscher, B.M. (2005). Model selection and inference: facts and
fiction. *Econometric Theory*, 21(1), 21–59.

White, H. (2000). A reality check for data snooping. *Econometrica*, 68(5),
1097–1126.